# Car Price Prediction
Assignment: predict used-car `Price` using **Linear Regression** and **Random Forest Regressor**, trained on the cleaned car dataset (`clean_car_dataset.csv`).

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


## 1. Load Dataset

In [2]:
df = pd.read_csv('clean_car_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (140, 12)


    Price  Odometer_km     Doors  ...  Km_per_year  Is_Urban  LogPrice
0  1500.0     0.128390  0.254091  ...    -0.615631         1  7.313887
1  4171.0    -0.044709  0.254091  ...     0.070446         0  8.336151
2  5331.0    -0.440923  0.254091  ...    -0.267993         0  8.581482
3  1500.0     0.203135  0.254091  ...    -0.587024         0  7.313887
4  1500.0    -0.044709 -0.931668  ...     1.738196         1  7.313887

[5 rows x 12 columns]

## 2. Prepare Features & Target
- Target (`y`) = `Price`
- Features (`X`) = all columns except `Price` and `LogPrice` (the latter is a log-transformed duplicate of the target and must be excluded to avoid leakage).

In [3]:
y = df['Price']
X = df.drop(columns=['Price', 'LogPrice'])

print('Feature columns:', list(X.columns))
print('Target: Price')
print('X shape:', X.shape, '| y shape:', y.shape)

Feature columns: ['Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_City', 'Location_Rural', 'Location_Suburb', 'CarAge', 'Km_per_year', 'Is_Urban']
Target: Price
X shape: (140, 10) | y shape: (140,)


## 3. Split Data
80% training / 20% testing, `random_state=42`.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape[0])
print('Test size :', X_test.shape[0])

Train size: 112
Test size : 28


## 4. Train Models

In [5]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

LinearRegression()

In [6]:
# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

## 5. Evaluate Performance

In [7]:
def evaluate_model(model, X_test, y_test, name):
    """Print R², MAE, MSE, and RMSE for a fitted regression model."""
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print(f'{name} Performance:')
    print(f'  R²   : {r2:.2f}')
    print(f'  MAE  : {mae:,.0f}')
    print(f'  MSE  : {mse:,.0f}')
    print(f'  RMSE : {rmse:,.0f}')

    return {'r2': r2, 'mae': mae, 'mse': mse, 'rmse': rmse}

In [8]:
lin_metrics = evaluate_model(lin_reg, X_test, y_test, 'Linear Regression')
print()
rf_metrics = evaluate_model(rf_reg, X_test, y_test, 'Random Forest')

Linear Regression Performance:
  R²   : 0.44
  MAE  : 1,428
  MSE  : 3,755,299
  RMSE : 1,938

Random Forest Performance:
  R²   : 0.27
  MAE  : 1,205
  MSE  : 4,831,226
  RMSE : 2,198


## 6. Sanity Check
Pick one row from the test set and compare the actual price with each model's prediction.

In [9]:
i = 0
sample = X_test.iloc[[i]]

actual_price = y_test.iloc[i]
lin_pred = lin_reg.predict(sample)[0]
rf_pred = rf_reg.predict(sample)[0]

print('Sample test row (features):')
print(sample.to_string(index=False))
print()
print(f'Actual Price               : {actual_price:,.2f}')
print(f'Linear Regression Predicted: {lin_pred:,.2f}')
print(f'Random Forest Predicted    : {rf_pred:,.2f}')

Sample test row (features):
 Odometer_km    Doors  Accidents     Year  Location_City  Location_Rural  Location_Suburb    CarAge  Km_per_year  Is_Urban
   -0.802507 0.254091   0.316968 0.381062              1               0                0 -0.381062    -0.456483         1

Actual Price               : 4,379.00
Linear Regression Predicted: 5,040.46
Random Forest Predicted    : 4,522.48


## 7. Metrics Summary Table

In [10]:
summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R2': [lin_metrics['r2'], rf_metrics['r2']],
    'MAE': [lin_metrics['mae'], rf_metrics['mae']],
    'MSE': [lin_metrics['mse'], rf_metrics['mse']],
    'RMSE': [lin_metrics['rmse'], rf_metrics['rmse']],
})
summary

               Model        R2          MAE           MSE         RMSE
0  Linear Regression  0.435800  1428.054663  3.755299e+06  1937.859445
1      Random Forest  0.274151  1204.807589  4.831226e+06  2198.005026